<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/08-window-functions/00-window-spec.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 8.0 — The window spec: partitionBy, orderBy, frames

One mechanism: an aggregate that collapses nothing. We build the
spec layer by layer on `orders.csv` and watch the meaning change —
especially the default-frame trap when orderBy is added.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("8.0-window-spec")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

orders = (
    spark.read.csv(f"{DATA_DIR}/orders.csv", header=True, inferSchema=True)
    .withColumn("revenue", F.round(col("quantity") * col("unit_price"), 2))
)
orders.select("order_id", "customer_id", "order_date", "revenue").show(5)

## Layer 1 — partitionBy: the aggregate that collapses nothing

groupBy destroys the order rows; the window keeps them and annotates.
And the question that needed a join yesterday is now one expression.

In [ ]:
w_cust = Window.partitionBy("customer_id")

with_total = orders.withColumn("cust_total", F.sum("revenue").over(w_cust))
(with_total
 .withColumn("share_of_cust", F.round(col("revenue") / col("cust_total"), 2))
 .select("order_id", "customer_id", "revenue", "cust_total", "share_of_cust")
 .orderBy("customer_id", "order_id")
 .show(8))

In [ ]:
# the pre-window idiom this replaces: aggregate + self-join. Same result,
# one more shuffle, three more lines. Recognize it in old code and refactor.
agg = orders.groupBy("customer_id").agg(F.sum("revenue").alias("cust_total"))
via_join = orders.join(agg, "customer_id")
print(via_join.count() == with_total.count())

## Layer 2 — orderBy... and the default-frame trap

Same F.sum. Adding orderBy silently flips the frame to
rangeBetween(unboundedPreceding, currentRow) — a RUNNING sum.

In [ ]:
w_plain = Window.partitionBy("customer_id")
w_ordered = Window.partitionBy("customer_id").orderBy("order_date")

(orders
 .withColumn("sum_no_order", F.sum("revenue").over(w_plain))
 .withColumn("sum_with_order", F.sum("revenue").over(w_ordered))   # surprise!
 .select("order_id", "customer_id", "order_date", "revenue",
         "sum_no_order", "sum_with_order")
 .orderBy("customer_id", "order_date")
 .show(9))

## Layer 3 — frames: rowsBetween vs rangeBetween

Physical vs logical distance. orders.csv has several orders per
date, so date-ordered ties make the two visibly diverge: ROWS climbs
within a day (in unstable tie order); RANGE treats each day as a block.

In [ ]:
w_rows = (Window.orderBy(F.col("order_date"))
          .rowsBetween(Window.unboundedPreceding, Window.currentRow))
w_range = (Window.orderBy(F.col("order_date"))
           .rangeBetween(Window.unboundedPreceding, Window.currentRow))

(orders
 .withColumn("running_rows", F.round(F.sum("revenue").over(w_rows), 2))
 .withColumn("running_range", F.round(F.sum("revenue").over(w_range), 2))
 .select("order_id", "order_date", "revenue", "running_rows", "running_range")
 .orderBy("order_date", "order_id")
 .show(10))
# every order sharing a date shows the SAME running_range; running_rows
# differs row to row — and its within-day order isn't guaranteed between runs

## What a window costs: read the plan

`Window` node fed by Exchange(hashpartitioning) + Sort — the same
shuffle currency as Module 7 joins. Two specs with the same
partition+order share one shuffle; change the partition key and you
buy a second one.

In [ ]:
same_spec = (orders
             .withColumn("a", F.sum("revenue").over(w_ordered))
             .withColumn("b", F.max("revenue").over(w_ordered)))
same_spec.explain()   # ONE Exchange, ONE Window node with two expressions

In [ ]:
two_specs = (orders
             .withColumn("a", F.sum("revenue").over(w_ordered))
             .withColumn("b", F.sum("revenue").over(Window.partitionBy("country"))))
two_specs.explain()   # TWO Exchanges — each partitioning is its own shuffle

## The empty-partitionBy warning, live

Legal, sometimes right, never scalable: all rows to ONE task.
Watch the log line WindowExec prints.

In [ ]:
w_global = Window.orderBy("order_date")   # no partitionBy
orders.withColumn("global_running", F.sum("revenue").over(w_global)).collect()
# console: WARN WindowExec: No Partition Defined for Window operation!
# 41 rows: harmless. 41 billion: the whole table on one core.

## Exercises

1. Add `pct_of_country`: each order's share of its ship-country's
   total revenue (null countries form their own group — decide and
   document what to do with them).
2. Predict before running: `F.count("*").over(Window.partitionBy(
   "customer_id").orderBy("order_date"))` on customer c001's three
   orders — what values, and why isn't it 3,3,3? Fix it to be 3,3,3
   WITHOUT removing the orderBy.
3. Express "average revenue of each customer's OTHER orders" (exclude
   the current row) with one window + arithmetic — no self-join.
   Hint: sum, count, subtract.
4. Take the two_specs plan and reorder the withColumns so the
   same-partitioning windows are adjacent. Does the number of
   Exchanges change? What does that tell you about how you should
   group window columns in a long pipeline?

In [ ]:
# your exercise code here